# Аналіз затримок авіарейсів у США у 2015 році

**Курсова робота з дисципліни «Мови програмування для аналізу даних»**
Виконав: Барановський Петро Миколайович, група 401-ТН
Керівник: ст. викл. Гайтан О. М.

**Мета:** виявити закономірності затримок і скасувань внутрішніх авіарейсів США у 2015 році залежно від авіакомпанії, аеропорту, сезону та часу доби.

**Дослідницькі питання**

| № | Питання |
|---|---|
| П1 | Які авіакомпанії та аеропорти мають найвищу частку затриманих рейсів і найбільшу середню затримку? |
| П2 | Як змінюються затримки за місяцями, днями тижня та часом доби вильоту? |
| П3 | Які причини дають найбільший внесок у загальну тривалість затримок? |
| П4 | Чи пов’язана величина затримки з дальністю польоту? |
| П5 | Де і коли скасовується найбільше рейсів і з яких причин? |

**Дані:** «2015 Flight Delays and Cancellations», U.S. Department of Transportation, https://www.kaggle.com/datasets/usdot/flight-delays (ліцензія CC0).

**Структура ноутбука**
1. Налаштування середовища
2. Завантаження та первинне дослідження даних
3. Перевірка якості та попереднє опрацювання
4. Нові аналітичні ознаки
5. Об’єднання таблиць
6. Статистичний та аналітичний аналіз
7. Візуалізація
8. Експорт даних для Tableau Public
9. Висновки

## 1. Налаштування середовища

Підключаємо бібліотеки, задаємо стиль графіків і шляхи до папок проєкту. Ноутбук розрахований на запуск із папки `notebooks/`; вихідні CSV-файли мають лежати в `data/raw/` (див. README.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

# Шляхи до папок проєкту
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIG_DIR = PROJECT_DIR / "results" / "figures"
EXPORT_DIR = PROJECT_DIR / "results" / "exports"
TABLEAU_DIR = PROJECT_DIR / "tableau"
for folder in (PROCESSED_DIR, FIG_DIR, EXPORT_DIR, TABLEAU_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Рейс вважається затриманим, якщо прибув на 15+ хв пізніше розкладу (визначення BTS)
DELAY_THRESHOLD = 15

# Назви п'яти причин затримки (поля таблиці flights)
CAUSE_COLS = {
    "AIRLINE_DELAY": "Авіакомпанія",
    "LATE_AIRCRAFT_DELAY": "Пізнє прибуття літака",
    "AIR_SYSTEM_DELAY": "Авіасистема (NAS)",
    "WEATHER_DELAY": "Погода",
    "SECURITY_DELAY": "Безпека",
}


def save_fig(fig, name):
    """Зберігає графік у results/figures для пояснювальної записки."""
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")


print("Папка проєкту:", PROJECT_DIR)
print("pandas", pd.__version__, "| seaborn", sns.__version__)

## 2. Завантаження та первинне дослідження даних

Набір складається з трьох таблиць: `flights` (рейси), `airlines` (авіакомпанії), `airports` (аеропорти).

З таблиці `flights` завантажуємо 25 із 31 стовпця: технічні моменти руху літака (`WHEELS_OFF`, `WHEELS_ON`, фактичний час вильоту і прибуття тощо) для дослідницьких питань не потрібні, а вже враховані в полях затримок. Це зменшує обсяг пам’яті приблизно на чверть.

Коди аеропортів читаємо як рядки (`dtype=str`), бо частина кодів за жовтень записана цифрами, і без цього pandas змішав би типи.

In [ ]:
FLIGHT_COLS = [
    "YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE", "DEPARTURE_DELAY",
    "TAXI_OUT", "SCHEDULED_TIME", "ELAPSED_TIME", "AIR_TIME", "DISTANCE", "ARRIVAL_DELAY",
    "DIVERTED", "CANCELLED", "CANCELLATION_REASON", *CAUSE_COLS,
]
TEXT_COLS = {c: "string" for c in ["AIRLINE", "TAIL_NUMBER", "ORIGIN_AIRPORT",
                                    "DESTINATION_AIRPORT", "CANCELLATION_REASON"]}

flights = pd.read_csv(RAW_DIR / "flights.csv", usecols=FLIGHT_COLS, dtype=TEXT_COLS)
airlines = pd.read_csv(RAW_DIR / "airlines.csv")
airports = pd.read_csv(RAW_DIR / "airports.csv")

for name, df in {"flights": flights, "airlines": airlines, "airports": airports}.items():
    print(f"{name:9}: {df.shape[0]:>10,} рядків × {df.shape[1]} стовпців")

In [ ]:
flights.head()

In [ ]:
flights.info(memory_usage="deep")

In [ ]:
airlines.head(14)

In [ ]:
airports.head()

**Описова статистика числових полів.** Звертаємо увагу на мінімуми та максимуми затримок: від’ємні значення означають ранній виліт/прибуття, дуже великі – потенційні викиди.

In [ ]:
flights.describe().T

In [ ]:
# Кількість унікальних значень категоріальних ознак
flights[["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "TAIL_NUMBER",
         "CANCELLATION_REASON"]].nunique().to_frame("Унікальних значень")

In [ ]:
# Скільки рейсів виконала кожна авіакомпанія
flights["AIRLINE"].value_counts().to_frame("Рейсів")

**Висновок до етапу 2** _(заповнити після запуску)_: кількість рейсів і стовпців; які ознаки числові, категоріальні, часові; які поля будуть ключами для об’єднання (`AIRLINE` → `airlines.IATA_CODE`, `ORIGIN_AIRPORT` → `airports.IATA_CODE`); які поля використаємо для групування (авіакомпанія, аеропорт, місяць, день тижня, година).

## 3. Перевірка якості та попереднє опрацювання

### 3.1. Пропущені значення

In [ ]:
rows_before = len(flights)
missing_before = (flights.isna().mean() * 100).round(2)
missing_before[missing_before > 0].sort_values(ascending=False).to_frame("Пропуски, %")

Перевіряємо гіпотезу, що пропуски **не випадкові**: у скасованих рейсів немає фактичного часу й затримок, у переадресованих – затримки прибуття. Якщо це так, пропуски логічні, і видаляти чи заповнювати їх не можна – інакше скасований рейс виглядав би як рейс «без затримки».

In [ ]:
is_cancelled = flights["CANCELLED"] == 1
is_diverted = flights["DIVERTED"] == 1
is_completed = ~is_cancelled & ~is_diverted

delay_cols = ["DEPARTURE_DELAY", "ARRIVAL_DELAY", "AIR_TIME"]
pd.DataFrame({
    "Скасовані": flights.loc[is_cancelled, delay_cols].isna().mean() * 100,
    "Переадресовані": flights.loc[is_diverted, delay_cols].isna().mean() * 100,
    "Виконані": flights.loc[is_completed, delay_cols].isna().mean() * 100,
}).round(2).rename_axis("Пропуски, % у групі")

In [ ]:
# Поля причин заповнюються лише для рейсів, затриманих на 15+ хв
has_cause = flights[list(CAUSE_COLS)].notna().all(axis=1)
pd.crosstab(
    flights["ARRIVAL_DELAY"] >= DELAY_THRESHOLD, has_cause,
    rownames=["Затримка прибуття ≥ 15 хв"], colnames=["Причини заповнені"],
)

### 3.2. Дублікати
Перевіряємо повні дублікати рядків і дублікати за логічним ключем рейсу (дата + авіакомпанія + номер рейсу + аеропорт вильоту + час за розкладом).

In [ ]:
FLIGHT_KEY = ["YEAR", "MONTH", "DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT",
              "SCHEDULED_DEPARTURE"]
n_full_dup = flights.duplicated().sum()
n_key_dup = flights.duplicated(subset=FLIGHT_KEY).sum()
print(f"Повних дублікатів: {n_full_dup:,}")
print(f"Дублікатів за ключем рейсу: {n_key_dup:,}")

In [ ]:
flights = flights.drop_duplicates().reset_index(drop=True)
print(f"Видалено повних дублікатів: {rows_before - len(flights):,}")

# Після видалення рядків маски потрібно перерахувати
is_cancelled = flights["CANCELLED"] == 1
is_diverted = flights["DIVERTED"] == 1
is_completed = ~is_cancelled & ~is_diverted

### 3.3. Коди аеропортів за жовтень
У частині записів код аеропорту – п’ятизначне число замість трилітерного коду IATA. Такі рейси не з’єднаються з таблицею `airports`. Перевіряємо, скільки їх і в які місяці.

In [ ]:
numeric_code = flights["ORIGIN_AIRPORT"].str.fullmatch(r"\d+").fillna(False).astype(bool)
print(f"Рейсів із числовим кодом аеропорту: {numeric_code.sum():,} "
      f"({numeric_code.mean():.1%} від усіх)")
flights.loc[numeric_code, "MONTH"].value_counts().sort_index().to_frame("Рейсів")

**Рішення:** рядки не видаляємо – для аналізу авіакомпаній, причин і динаміки вони коректні. Для аналізу на рівні аеропортів ці рейси автоматично випадуть після об’єднання (пропуск у назві аеропорту). Це обмеження фіксуємо у висновках.

### 3.4. Некоректні та суперечливі значення

In [ ]:
logic_checks = pd.Series({
    "Дальність ≤ 0": (flights["DISTANCE"] <= 0).sum(),
    "Час у повітрі ≤ 0": (flights["AIR_TIME"] <= 0).sum(),
    "Скасований без причини": (is_cancelled & flights["CANCELLATION_REASON"].isna()).sum(),
    "Не скасований, але з причиною": (~is_cancelled & flights["CANCELLATION_REASON"].notna()).sum(),
    "Скасований і переадресований одночасно": (is_cancelled & is_diverted).sum(),
    "Затримка прибуття > 24 год": (flights["ARRIVAL_DELAY"] > 24 * 60).sum(),
}, name="Кількість записів")
logic_checks.to_frame()

### 3.5. Викиди
Шукаємо викиди затримки прибуття методом міжквартильного розмаху (IQR): викидом вважаємо значення більше за Q3 + 1,5·IQR.

In [ ]:
arr = flights.loc[is_completed, "ARRIVAL_DELAY"]
q1, q3 = arr.quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
outliers = arr > upper_fence
print(f"Q1 = {q1:.0f} хв, Q3 = {q3:.0f} хв, IQR = {iqr:.0f} хв, верхня межа = {upper_fence:.0f} хв")
print(f"Викидів: {outliers.sum():,} ({outliers.mean():.1%} виконаних рейсів)")
print(f"Затримок понад 3 год: {(arr > 180).sum():,}; понад 10 год: {(arr > 600).sum():,}")

**Рішення щодо викидів:** не видаляємо. Великі затримки – реальна частина процесу (снігові бурі, поломки), і саме вони найбільше шкодять пасажирам. Щоб викиди не спотворювали висновки, поряд із середнім використовуємо **медіану**, а на графіках обмежуємо вісь.

### 3.6. Перетворення типів і заповнення пропусків
- дата рейсу з трьох полів → `datetime` (`to_datetime`);
- `CANCELLED`, `DIVERTED`: 0/1 → логічний тип (`astype`);
- коди причин скасування A–D → зрозумілі назви (`replace`);
- пропуски в полях причин затримки → 0 хвилин (`fillna`): BTS указує причини лише для затриманих рейсів, тож для інших рейсів на причину припадає 0 хвилин. Пропуски в самих затримках не чіпаємо (див. 3.1).

In [ ]:
CANCEL_REASONS = {"A": "Авіакомпанія", "B": "Погода", "C": "Авіасистема (NAS)", "D": "Безпека"}

flights["FLIGHT_DATE"] = pd.to_datetime(
    flights[["YEAR", "MONTH", "DAY"]].rename(columns=str.lower)
)
flights[["CANCELLED", "DIVERTED"]] = flights[["CANCELLED", "DIVERTED"]].astype(bool)
flights["CANCELLATION_REASON"] = flights["CANCELLATION_REASON"].replace(CANCEL_REASONS)
flights[list(CAUSE_COLS)] = flights[list(CAUSE_COLS)].fillna(0)

flights[["FLIGHT_DATE", "CANCELLED", "DIVERTED", "CANCELLATION_REASON"]].dtypes

In [ ]:
# Підсумок очищення
missing_after = (flights.isna().mean() * 100).round(2)
cleaning_summary = pd.DataFrame({
    "Пропуски до, %": missing_before,
    "Пропуски після, %": missing_after,
}).loc[lambda d: d.max(axis=1) > 0]
print(f"Рядків до очищення: {rows_before:,}; після: {len(flights):,}")
cleaning_summary

## 4. Нові аналітичні ознаки

| Ознака | Як обчислюється | Навіщо |
|---|---|---|
| `IS_COMPLETED` | не скасований і не переадресований | лише для таких рейсів коректно рахувати затримку прибуття |
| `IS_DELAYED` | затримка прибуття ≥ 15 хв | головний показник: частка затриманих рейсів |
| `DELAY_CATEGORY` | інтервали затримки (`cut`) | структура запізнень, а не лише середнє |
| `ARRIVAL_LATE_MIN` | затримка, де ранні прибуття = 0 (`where`) | «чисті» хвилини запізнення без компенсації ранніми рейсами |
| `DEP_HOUR`, `TIME_OF_DAY` | година вильоту за розкладом, частина доби | П2: залежність від часу доби |
| `DAY_NAME`, `SEASON` | назва дня тижня, сезон | П2: тижнева й сезонна динаміка |
| `DISTANCE_KM`, `DISTANCE_GROUP` | милі → км; квартильні групи (`qcut`) | П4: зв’язок із дальністю |

In [ ]:
DAY_NAMES = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Нд"]
SEASONS = {12: "Зима", 1: "Зима", 2: "Зима", 3: "Весна", 4: "Весна", 5: "Весна",
           6: "Літо", 7: "Літо", 8: "Літо", 9: "Осінь", 10: "Осінь", 11: "Осінь"}

flights = flights.assign(
    IS_COMPLETED=~flights["CANCELLED"] & ~flights["DIVERTED"],
    IS_DELAYED=flights["ARRIVAL_DELAY"] >= DELAY_THRESHOLD,
    DELAY_CATEGORY=pd.cut(
        flights["ARRIVAL_DELAY"], bins=[-np.inf, 1, DELAY_THRESHOLD, 61, np.inf], right=False,
        labels=["Вчасно або раніше", "1–14 хв", "15–60 хв", "Понад 60 хв"],
    ),
    ARRIVAL_LATE_MIN=flights["ARRIVAL_DELAY"].where(flights["ARRIVAL_DELAY"] > 0, 0),
    DEP_HOUR=(flights["SCHEDULED_DEPARTURE"] // 100) % 24,
    DAY_NAME=pd.Categorical(flights["DAY_OF_WEEK"].map(dict(enumerate(DAY_NAMES, 1))),
                            categories=DAY_NAMES, ordered=True),
    SEASON=pd.Categorical(flights["MONTH"].map(SEASONS),
                          categories=["Зима", "Весна", "Літо", "Осінь"], ordered=True),
    DISTANCE_KM=(flights["DISTANCE"] * 1.609).round(),
)
# Для скасованих рейсів хвилини запізнення не визначені
flights.loc[~flights["IS_COMPLETED"], "ARRIVAL_LATE_MIN"] = np.nan

flights["TIME_OF_DAY"] = pd.cut(
    flights["DEP_HOUR"], bins=[-1, 5, 11, 17, 23],
    labels=["Ніч (0–5)", "Ранок (6–11)", "День (12–17)", "Вечір (18–23)"],
)
flights["DISTANCE_GROUP"] = pd.qcut(
    flights["DISTANCE_KM"], q=4, labels=["Короткі", "Середні", "Довгі", "Дуже довгі"],
)

flights[["ARRIVAL_DELAY", "IS_DELAYED", "DELAY_CATEGORY", "ARRIVAL_LATE_MIN", "DEP_HOUR",
         "TIME_OF_DAY", "DAY_NAME", "SEASON", "DISTANCE_KM", "DISTANCE_GROUP"]].head(8)

In [ ]:
# Межі груп дальності (квартилі), км
flights.groupby("DISTANCE_GROUP", observed=True)["DISTANCE_KM"].agg(["min", "max", "size"])

## 5. Об’єднання таблиць

1. `flights.AIRLINE` → `airlines.IATA_CODE`: повна назва авіакомпанії.
2. `flights.ORIGIN_AIRPORT` → `airports.IATA_CODE`: назва, місто, штат і координати аеропорту вильоту (потрібні для карти в Tableau).

Використовуємо `how="left"`, щоб не втратити жодного рейсу, і `validate="many_to_one"` – pandas перевірить, що в довідниках немає повторних кодів.

In [ ]:
airlines_ref = airlines.rename(columns={"IATA_CODE": "AIRLINE", "AIRLINE": "AIRLINE_NAME"})
airlines_ref["AIRLINE"] = airlines_ref["AIRLINE"].astype("string")

origin_ref = airports.rename(columns={
    "IATA_CODE": "ORIGIN_AIRPORT", "AIRPORT": "ORIGIN_NAME", "CITY": "ORIGIN_CITY",
    "STATE": "ORIGIN_STATE", "LATITUDE": "ORIGIN_LAT", "LONGITUDE": "ORIGIN_LON",
}).drop(columns="COUNTRY")
origin_ref["ORIGIN_AIRPORT"] = origin_ref["ORIGIN_AIRPORT"].astype("string")

rows_before_merge = len(flights)
flights = (
    flights
    .merge(airlines_ref, on="AIRLINE", how="left", validate="many_to_one")
    .merge(origin_ref, on="ORIGIN_AIRPORT", how="left", validate="many_to_one")
)
assert len(flights) == rows_before_merge, "Об'єднання не повинно змінювати кількість рейсів"

merge_quality = pd.Series({
    "Рейсів без назви авіакомпанії": flights["AIRLINE_NAME"].isna().sum(),
    "Рейсів без даних аеропорту": flights["ORIGIN_NAME"].isna().sum(),
    "  з них із числовим кодом": (flights["ORIGIN_NAME"].isna()
                                  & flights["ORIGIN_AIRPORT"].str.fullmatch(r"\d+").fillna(False)).sum(),
    "Рейсів без координат аеропорту": flights["ORIGIN_LAT"].isna().sum(),
}, name="Кількість")
merge_quality.to_frame()

Аналіз затримок виконуємо на **виконаних** рейсах (`done`), аналіз скасувань – на всіх (`flights`).

In [ ]:
done = flights[flights["IS_COMPLETED"]].copy()
print(f"Усього рейсів: {len(flights):,}; виконаних: {len(done):,}; "
      f"скасованих: {flights['CANCELLED'].sum():,}; переадресованих: {flights['DIVERTED'].sum():,}")

## 6. Статистичний та аналітичний аналіз

### 6.1. Описова статистика і ключові показники

In [ ]:
num_cols = ["DEPARTURE_DELAY", "ARRIVAL_DELAY", "TAXI_OUT", "AIR_TIME", "DISTANCE_KM"]
desc = done[num_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T
desc["skew"] = done[num_cols].skew()
desc

In [ ]:
kpi = pd.Series({
    "Рейсів усього": len(flights),
    "Частка затриманих (≥15 хв), %": done["IS_DELAYED"].mean() * 100,
    "Частка скасованих, %": flights["CANCELLED"].mean() * 100,
    "Середня затримка прибуття, хв": done["ARRIVAL_DELAY"].mean(),
    "Медіана затримки прибуття, хв": done["ARRIVAL_DELAY"].median(),
    "Середнє запізнення затриманого рейсу, хв": done.loc[done["IS_DELAYED"], "ARRIVAL_DELAY"].mean(),
}, name="Значення")
kpi.to_frame()

In [ ]:
# Структура рейсів за категоріями затримки
(done["DELAY_CATEGORY"].value_counts(normalize=True, sort=False) * 100).to_frame("Частка, %")

**Інтерпретація** _(заповнити)_: порівняти середнє й медіану – якщо середнє значно більше, розподіл правоасиметричний (підтверджує `skew`), і типовий рейс затримується менше, ніж показує середнє.

### 6.2. Авіакомпанії: групування, агрегування, ранжування (П1)

In [ ]:
delay_by_airline = done.groupby("AIRLINE_NAME").agg(
    flights=("ARRIVAL_DELAY", "size"),
    delayed_pct=("IS_DELAYED", "mean"),
    mean_delay=("ARRIVAL_DELAY", "mean"),
    median_delay=("ARRIVAL_DELAY", "median"),
    std_delay=("ARRIVAL_DELAY", "std"),
)
cancel_by_airline = flights.groupby("AIRLINE_NAME")["CANCELLED"].mean().rename("cancelled_pct")

airline_stats = delay_by_airline.join(cancel_by_airline)
airline_stats[["delayed_pct", "cancelled_pct"]] *= 100
airline_stats["rank_delayed"] = airline_stats["delayed_pct"].rank(ascending=False, method="min").astype(int)
airline_stats["rank_cancelled"] = airline_stats["cancelled_pct"].rank(ascending=False, method="min").astype(int)
airline_stats = airline_stats.sort_values("rank_delayed")
airline_stats

**Інтерпретація** _(заповнити)_: лідери й аутсайдери за часткою затриманих; чи збігаються рейтинги за затримками й скасуваннями; у кого найбільший розкид (`std_delay`).

### 6.3. Аеропорти: лідери та аутсайдери (П1)
Щоб маленькі аеропорти з кількома рейсами не потрапили в рейтинг випадково, беремо лише аеропорти, на які припадає щонайменше 0,2 % усіх вильотів.

In [ ]:
MIN_AIRPORT_FLIGHTS = max(100, int(len(done) * 0.002))

airport_stats = (
    done.dropna(subset=["ORIGIN_NAME"])
    .groupby(["ORIGIN_AIRPORT", "ORIGIN_CITY", "ORIGIN_STATE"])
    .agg(flights=("IS_DELAYED", "size"), delayed_pct=("IS_DELAYED", "mean"),
         mean_delay=("ARRIVAL_DELAY", "mean"), median_taxi_out=("TAXI_OUT", "median"))
    .query("flights >= @MIN_AIRPORT_FLIGHTS")
)
airport_stats["delayed_pct"] *= 100
print(f"Поріг: {MIN_AIRPORT_FLIGHTS:,} вильотів; аеропортів у рейтингу: {len(airport_stats)}")

In [ ]:
print("Найбільша частка затриманих рейсів:")
display(airport_stats.nlargest(10, "delayed_pct"))
print("Найменша частка затриманих рейсів:")
display(airport_stats.nsmallest(10, "delayed_pct"))

### 6.4. Зведені таблиці: день тижня × час доби, сезон × частина доби (П2)

In [ ]:
pivot_day_hour = done.pivot_table(
    index="DAY_NAME", columns="DEP_HOUR", values="IS_DELAYED", aggfunc="mean", observed=True,
) * 100
pivot_day_hour.round(1)

In [ ]:
pivot_season_tod = done.pivot_table(
    index="SEASON", columns="TIME_OF_DAY", values="ARRIVAL_DELAY",
    aggfunc="mean", observed=True, margins=True, margins_name="Усього",
)
pivot_season_tod.round(1)

**Інтерпретація** _(заповнити)_: у яку частину доби затримок більше; чи наростають вони до вечора (ланцюговий ефект: пізнє прибуття літака переноситься на наступні рейси); найгірший сезон.

### 6.5. Структура причин затримок (П3)

In [ ]:
cause_minutes = done[list(CAUSE_COLS)].sum().rename(CAUSE_COLS)
cause_share = (cause_minutes / cause_minutes.sum() * 100).sort_values(ascending=False)
pd.DataFrame({"Хвилин": cause_minutes, "Частка, %": cause_share}).sort_values("Частка, %", ascending=False)

In [ ]:
causes_by_airline = done.groupby("AIRLINE_NAME")[list(CAUSE_COLS)].sum().rename(columns=CAUSE_COLS)
causes_by_airline_pct = causes_by_airline.div(causes_by_airline.sum(axis=1), axis=0) * 100
causes_by_airline_pct = causes_by_airline_pct.loc[airline_stats.index]  # той самий порядок, що в рейтингу
causes_by_airline_pct.round(1)

### 6.6. Скасування: місяць × причина (П5)

In [ ]:
cancel_crosstab = pd.crosstab(
    flights.loc[flights["CANCELLED"], "MONTH"],
    flights.loc[flights["CANCELLED"], "CANCELLATION_REASON"],
    margins=True, margins_name="Усього",
)
cancel_crosstab

In [ ]:
cancel_by_airport = (
    flights.dropna(subset=["ORIGIN_NAME"])
    .groupby(["ORIGIN_AIRPORT", "ORIGIN_CITY"])
    .agg(flights=("CANCELLED", "size"), cancelled_pct=("CANCELLED", "mean"))
    .query("flights >= @MIN_AIRPORT_FLIGHTS")
)
cancel_by_airport["cancelled_pct"] *= 100
cancel_by_airport.nlargest(10, "cancelled_pct")

### 6.7. Взаємозв’язки (П4)
Рахуємо коефіцієнти кореляції Пірсона (лінійний зв’язок) і Спірмена (монотонний, стійкий до викидів). Кореляція не означає причинно-наслідкового зв’язку.

In [ ]:
corr_cols = ["DEPARTURE_DELAY", "ARRIVAL_DELAY", "TAXI_OUT", "AIR_TIME", "DISTANCE_KM", "DEP_HOUR"]
corr_pearson = done[corr_cols].corr(method="pearson")
corr_pearson.round(2)

In [ ]:
dist_corr = pd.Series({
    "Пірсон": done["DISTANCE_KM"].corr(done["ARRIVAL_DELAY"]),
    "Спірмен": done["DISTANCE_KM"].corr(done["ARRIVAL_DELAY"], method="spearman"),
}, name="Кореляція: дальність ~ затримка прибуття")
display(dist_corr.to_frame())

done.groupby("DISTANCE_GROUP", observed=True).agg(
    delayed_pct=("IS_DELAYED", lambda s: s.mean() * 100),
    mean_delay=("ARRIVAL_DELAY", "mean"),
    median_delay=("ARRIVAL_DELAY", "median"),
)

### 6.8. Динаміка в часі (П2)

In [ ]:
monthly = flights.groupby("MONTH").agg(
    flights=("CANCELLED", "size"),
    cancelled_pct=("CANCELLED", "mean"),
).join(
    done.groupby("MONTH").agg(delayed_pct=("IS_DELAYED", "mean"), mean_delay=("ARRIVAL_DELAY", "mean"))
)
monthly[["cancelled_pct", "delayed_pct"]] *= 100
monthly

In [ ]:
# Щоденна частка затриманих і 7-денне ковзне середнє
daily = done.set_index("FLIGHT_DATE").resample("D")["IS_DELAYED"].mean() * 100
daily_rolling = daily.rolling(window=7, center=True).mean()

print("П'ять найгірших днів року:")
daily.nlargest(5).round(1).to_frame("Затриманих, %")

## 7. Візуалізація

Вісім графіків, кожен відповідає на одне з дослідницьких питань. Усі графіки зберігаються в `results/figures/`.

### Рисунок 1. Розподіл затримки прибуття

In [ ]:
fig, ax = plt.subplots()
sns.histplot(done["ARRIVAL_DELAY"].clip(-60, 180), bins=60, ax=ax, color="#4C72B0")
ax.axvline(done["ARRIVAL_DELAY"].median(), color="black", ls="--",
           label=f"Медіана: {done['ARRIVAL_DELAY'].median():.0f} хв")
ax.axvline(DELAY_THRESHOLD, color="#C44E52", ls="-", label="Поріг затримки: 15 хв")
ax.set(title="Розподіл затримки прибуття (обрізано до −60…180 хв)",
       xlabel="Затримка прибуття, хв", ylabel="Кількість рейсів")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.legend()
save_fig(fig, "01_delay_distribution")
plt.show()

**Інтерпретація** _(заповнити)_: форма розподілу, де більшість рейсів, наскільки довгий правий «хвіст».

### Рисунок 2. Частка затриманих рейсів за авіакомпаніями

In [ ]:
plot_data = airline_stats.sort_values("delayed_pct")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_data.index, plot_data["delayed_pct"], color="#4C72B0")
ax.axvline(done["IS_DELAYED"].mean() * 100, color="black", ls="--", label="Середнє по всіх")
for y, v in enumerate(plot_data["delayed_pct"]):
    ax.text(v + 0.2, y, f"{v:.1f}%", va="center", fontsize=9)
ax.set(title="Частка рейсів, затриманих на 15+ хв, за авіакомпаніями",
       xlabel="Затримані рейси, %", ylabel="")
ax.set_xlim(0, plot_data["delayed_pct"].max() * 1.15)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.legend(loc="lower right", framealpha=1)
save_fig(fig, "02_airlines_delayed_pct")
plt.show()

**Інтерпретація** _(заповнити)_: найгірша і найкраща авіакомпанії, різниця у відсоткових пунктах.

### Рисунок 3. Розкид затримок за авіакомпаніями

In [ ]:
order = airline_stats.sort_values("median_delay").index
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=done, y="AIRLINE_NAME", x="ARRIVAL_DELAY", order=order,
            showfliers=False, color="#8DA0CB", ax=ax)
ax.axvline(DELAY_THRESHOLD, color="#C44E52", ls="--", label="Поріг 15 хв")
ax.set(title="Затримка прибуття за авіакомпаніями (без викидів)",
       xlabel="Затримка прибуття, хв", ylabel="")
ax.legend(loc="lower right")
save_fig(fig, "03_airlines_boxplot")
plt.show()

**Інтерпретація** _(заповнити)_: у кого найширша «коробка» (найменш передбачувані рейси); чи медіани в усіх нижчі за 15 хв.

### Рисунок 4. Динаміка затримок і скасувань за місяцями

In [ ]:
MONTH_NAMES = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
fig, ax = plt.subplots()
ax.plot(monthly.index, monthly["delayed_pct"], marker="o", label="Затримані (≥15 хв)")
ax.plot(monthly.index, monthly["cancelled_pct"], marker="s", label="Скасовані")
ax.set_xticks(range(1, 13), MONTH_NAMES)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set(title="Частка затриманих і скасованих рейсів за місяцями 2015 року",
       xlabel="Місяць", ylabel="Частка рейсів, %")
ax.legend()
save_fig(fig, "04_monthly_dynamics")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(daily.index, daily, color="#B0B0B0", lw=0.8, label="За день")
ax.plot(daily_rolling.index, daily_rolling, color="#4C72B0", lw=2, label="Ковзне середнє, 7 днів")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set(title="Щоденна частка затриманих рейсів", xlabel="Дата", ylabel="Затримані рейси, %")
ax.legend()
save_fig(fig, "04b_daily_dynamics")
plt.show()

**Інтерпретація** _(заповнити)_: пікові місяці (зима? літні відпустки?), найспокійніші місяці, окремі «чорні» дні.

### Рисунок 5. Частка затриманих рейсів: день тижня × година вильоту

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
sns.heatmap(pivot_day_hour, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Затримані рейси, %"})
ax.set(title="Частка затриманих рейсів за днем тижня та годиною вильоту за розкладом",
       xlabel="Година вильоту", ylabel="День тижня")
save_fig(fig, "05_heatmap_day_hour")
plt.show()

**Інтерпретація** _(заповнити)_: «найбезпечніші» години (зазвичай ранок) і найгірші (вечір); чи є різниця між буднями й вихідними.

### Рисунок 6. Структура причин затримок за авіакомпаніями

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
causes_by_airline_pct.iloc[::-1].plot.barh(stacked=True, ax=ax, width=0.8, colormap="Set2")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set(title="Частка хвилин затримки за причинами", xlabel="Частка хвилин затримки, %", ylabel="")
ax.set_xlim(0, 100)
ax.legend(title="Причина", bbox_to_anchor=(1.01, 1), loc="upper left")
save_fig(fig, "06_causes_by_airline")
plt.show()

**Інтерпретація** _(заповнити)_: яка причина домінує загалом (див. 6.5); у кого частка «вини авіакомпанії» найбільша; роль погоди.

### Рисунок 7. Взаємозв’язки числових ознак і дальність польоту

In [ ]:
sample = done.sample(n=min(10_000, len(done)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 1.1]})

sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title("Кореляційна матриця (Пірсон)")

sns.scatterplot(data=sample, x="DISTANCE_KM", y="ARRIVAL_DELAY", alpha=0.25, s=10, ax=axes[1])
axes[1].set_ylim(-60, 300)
axes[1].set(title=f"Дальність і затримка (вибірка {len(sample):,} рейсів)",
            xlabel="Дальність польоту, км", ylabel="Затримка прибуття, хв")
fig.tight_layout()
save_fig(fig, "07_correlations")
plt.show()

**Інтерпретація** _(заповнити)_: сильний зв’язок затримки вильоту й прибуття (очікувано); зв’язок із дальністю слабкий чи ні – з коефіцієнтом із 6.7.

### Рисунок 8. Скасування за місяцями та причинами

In [ ]:
fig, ax = plt.subplots()
cancel_plot = cancel_crosstab.drop(index="Усього", columns="Усього")
cancel_plot.plot.bar(stacked=True, ax=ax, colormap="Set2", width=0.8)
ax.set_xticklabels([MONTH_NAMES[m - 1] for m in cancel_plot.index], rotation=0)
ax.set(title="Кількість скасованих рейсів за місяцями та причинами",
       xlabel="Місяць", ylabel="Скасованих рейсів")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.legend(title="Причина")
save_fig(fig, "08_cancellations")
plt.show()

**Інтерпретація** _(заповнити)_: у які місяці пік скасувань і чи пояснюється він погодою.

## 8. Експорт даних для Tableau Public

Tableau Public погано працює з 5,8 млн рядків, тому готуємо агреговану таблицю: один рядок – місяць × авіакомпанія × аеропорт вильоту. У ній лише **суми й кількості**, а відсотки та середні Tableau рахує сам (`SUM(delayed) / SUM(completed)`), тому фільтри працюють коректно. Рейси з числовими кодами аеропортів у файл не потрапляють (немає координат).

Окремо зберігаємо підсумкові таблиці аналізу в `results/exports/`.

In [ ]:
tableau_df = (
    flights.dropna(subset=["ORIGIN_NAME", "ORIGIN_LAT"])
    .groupby(["MONTH", "AIRLINE", "AIRLINE_NAME", "ORIGIN_AIRPORT", "ORIGIN_NAME",
              "ORIGIN_CITY", "ORIGIN_STATE", "ORIGIN_LAT", "ORIGIN_LON"], observed=True)
    .agg(
        flights=("CANCELLED", "size"),
        cancelled=("CANCELLED", "sum"),
        diverted=("DIVERTED", "sum"),
        completed=("IS_COMPLETED", "sum"),
        delayed=("IS_DELAYED", "sum"),
        arr_delay_sum=("ARRIVAL_DELAY", "sum"),
        late_minutes_sum=("ARRIVAL_LATE_MIN", "sum"),
        **{f"{c.lower()}_sum": (c, "sum") for c in CAUSE_COLS},
    )
    .reset_index()
)
tableau_df["MONTH_NAME"] = tableau_df["MONTH"].map(dict(enumerate(MONTH_NAMES, 1)))

tableau_path = TABLEAU_DIR / "flights_tableau.csv"
tableau_df.to_csv(tableau_path, index=False, encoding="utf-8")
tableau_df.to_csv(PROCESSED_DIR / "flights_tableau.csv", index=False, encoding="utf-8")
print(f"Збережено {len(tableau_df):,} рядків у {tableau_path.relative_to(PROJECT_DIR)}")
print(f"Покрито рейсів: {tableau_df['flights'].sum():,} з {len(flights):,}")
tableau_df.head()

In [ ]:
exports = {
    "kpi.csv": kpi.to_frame(),
    "airline_stats.csv": airline_stats,
    "airport_stats.csv": airport_stats,
    "pivot_day_hour.csv": pivot_day_hour,
    "pivot_season_time_of_day.csv": pivot_season_tod,
    "causes_by_airline_pct.csv": causes_by_airline_pct,
    "cancellations_month_reason.csv": cancel_crosstab,
    "monthly.csv": monthly,
    "correlations.csv": corr_pearson,
}
for file_name, table in exports.items():
    table.to_csv(EXPORT_DIR / file_name, encoding="utf-8")
print("Збережено:", ", ".join(exports))

## 9. Висновки

_Заповнити після запуску на повних даних. Кожен пункт – з конкретними цифрами з таблиць і графіків вище._

1. **П1 – авіакомпанії та аеропорти:** …
2. **П2 – динаміка в часі:** …
3. **П3 – причини:** …
4. **П4 – дальність:** …
5. **П5 – скасування:** …

**Практичні рекомендації** (для пасажирів і перевізників): …

**Обмеження дослідження:**
- дані лише за 2015 рік і лише внутрішні рейси великих американських перевізників;
- рейси жовтня з числовими кодами аеропортів не увійшли в аналіз аеропортів;
- причини затримок указують самі авіакомпанії, що може впливати на їх розподіл;
- кореляції не доводять причинно-наслідкових зв’язків.